# Fine Tuning LLM for Basic Chat Generation

In [ ]:
pip install transformers

In [ ]:
!pip install -U torchao

In [ ]:
import transformers

# Dataset Format:
### {
  'input' : {'role': 'user' , 'mood':['flirty'  ,'friendly', 'serious'], 'text':'xxx'},
'output' : {role': 'bot' , 'mood':['flirty'  ,'friendly', 'serious'], 'text':'xxx'}
}

In [ ]:
from google.colab import files
data = files.upload()

In [ ]:
from datasets import load_dataset
dataset = load_dataset("text", data_files="chat_data.jsonl", split ="train")
print(dataset)
print(dataset[:2])

In [ ]:
import json

parsed = [json.loads(x) for x in dataset["text"]]

pairs = []

for i in range(len(parsed) - 1):
    if parsed[i]["role"] == "user" and parsed[i + 1]["role"] == "bot":
        pairs.append({
            "prompt": parsed[i]["text"],
            "completion": parsed[i + 1]["text"]
        })

pairs[:2]

In [ ]:
from datasets import Dataset
dataset = Dataset.from_list(pairs)
dataset

In [ ]:

len(dataset)

In [ ]:
dataset[0]

In [ ]:
from transformers import AutoTokenizer , AutoModelForCausalLM

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
from transformers import pipeline
llm = pipeline(model=model_name)
llm("Hey dear, how are you?")

In [ ]:
llm("Do you like cricket? 🙂")

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
def tokenize(data):
  prompt = data['prompt'] + "\n" + data['completion']
  tokenized = tokenizer(
      prompt,
      truncation = True,
      max_new_tokens=16
  )
  return tokenized
from transformers import DataCollatorForLanguageModeling



dataset = dataset.map(tokenize , remove_columns=dataset.column_names)
from pprint import pprint
pprint(dataset[:2])


In [ ]:
from peft import get_peft_model , LoraConfig , TaskType

config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model , config)

In [ ]:
from transformers import TrainingArguments , Trainer

train_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=2,

    learning_rate=2e-5,

    num_train_epochs=10,

    logging_steps=1,

    save_strategy="epoch",

    weight_decay=0.01,

    warmup_steps=5,

    fp16=True,

    report_to="none"
)

In [ ]:


# 2. RE-CREATE the collator so it picks up the change
from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 3. RE-INITIALIZE the trainer so it uses the new collator
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=dataset,
    data_collator=collator  # The refreshed collator
)

# 4. Now start
trainer.train()


In [ ]:
def test_model(user_input):
  inputs = tokenizer(user_input, return_tensors="pt").to('cuda')
  output = model.generate(**inputs, max_new_tokens=32,temperature=1.0)
  print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
test_model("Hello dear, how are you?")

In [ ]:
test_model("Do you like music? What do you listen to?")

In [ ]:
test_model("kesi ho baby?")

In [ ]:
test_model("We should really grab dinner somethimes you know 😁")

In [ ]:
test_model("What should i wear for tomorrow's party??")

In [ ]:
test_model("Who is elon musk?")

In [ ]:
test_model("Do you like cricket? 🙂")